# Sweet-Spot Threshold Analysis for Dataset Cleaning

This notebook investigates whether image-quality based cleaning thresholds improve the dataset before preprocessing and model training.

The goal is not to remove as many images as possible. Instead, the goal is to determine whether cleaning rules improve downstream proxy validation performance while preserving data diversity and class balance.

The analysis follows five principles:

1. Compute image-quality audit metrics for the full dataset.
2. Inspect metric distributions and visually examine extreme samples.
3. Evaluate threshold candidates using a fixed proxy validation protocol.
4. Compare cleaning presets using performance, retention, and class-balance stability.
5. Recommend a final cleaning policy based on practical evidence, not only numerical ranking.

This notebook is an experiment notebook. It supports the final image-classification pipeline, but it does not replace the main training notebook.

In [1]:
# =============================
# 0. ENVIRONMENT SETUP
# =============================

!pip -q install kagglehub imagehash opencv-python-headless scikit-image seaborn tqdm

import sys
import shutil
from pathlib import Path

PROJECT_ROOT = Path("/content/Submission").resolve()
MODULE_DIR = PROJECT_ROOT / "modules"

# If the folder already exists, use it directly.
# Otherwise, clone the public repository.
# IMPORTANT: Replace this URL once with your real GitHub repository URL.
REPO_URL = "https://github.com/kahn-29/252-MachineLearning-Assignment1.git"

if PROJECT_ROOT.exists():
    print("Using existing project folder:", PROJECT_ROOT)
else:
    print("Project folder not found. Cloning repository...")
    !git clone {REPO_URL} {PROJECT_ROOT}

if not MODULE_DIR.exists():
    raise FileNotFoundError(
        f"Cannot find modules folder at: {MODULE_DIR}\n"
        "Expected structure:\n"
        "/content/Submission/\n"
        "├── modules/\n"
        "│   ├── config_utils.py\n"
        "│   ├── data_utils.py\n"
        "│   ├── image_audit.py\n"
        "│   ├── cleaning.py\n"
        "│   ├── transforms.py\n"
        "│   ├── feature_extraction.py\n"
        "│   └── visualization.py\n"
    )

(MODULE_DIR / "__init__.py").touch()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Modules folder:", MODULE_DIR)
print("Python path updated.")

print("\nAvailable module files:")
for p in sorted(MODULE_DIR.glob("*.py")):
    print("-", p.name)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 4.8 MB/s eta 0:00:00
Project folder not found. Cloning repository...
Cloning into '/content/Submission'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 80 (delta 45), reused 76 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 3.78 MiB | 15.65 MiB/s, done.
Resolving deltas: 100% (45/45), done.
Project root: /content/Submission
Modules folder: /content/Submission/modules
Python path updated.

Available module files:
- __init__.py
- cleaning.py
- config_utils.py
- data_utils.py
- evaluation.py
- feature_extraction.py
- image_audit.py
- transforms.py
- visualization.py


# 1. Imports and Configuration

This section imports the reusable module layer and defines the experiment configuration.

The notebook only controls the experimental flow. Core logic such as dataset loading, image auditing, duplicate detection, feature extraction, cleaning rules, and visualization is handled by helper modules.

In [2]:
# =============================
# 1. IMPORTS AND CONFIGURATION
# =============================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import torch

from modules.config_utils import (
    set_seed,
    get_device,
    ensure_dirs,
    resolve_workspace,
)

from modules.data_utils import (
    resolve_dataset_root,
    build_raw_dataframe,
    summarize_class_distribution,
)

from modules.image_audit import (
    audit_dataframe,
    describe_audit_metrics,
)

from modules.cleaning import (
    mark_near_duplicates,
    build_cleaning_mask,
    apply_cleaning,
    summarize_cleaning,
    summarize_removal_reasons,
    evaluate_cleaning_retention,
)

from modules.transforms import get_hybrid_transform

from modules.feature_extraction import extract_features

from modules.visualization import (
    savefig as savefig_module,
    plot_metric_distribution,
    plot_metric_correlation_heatmap,
    plot_removed_examples,
    plot_before_after_cleaning,
)

warnings.filterwarnings("ignore")

SEED = 42
set_seed(SEED)

CONFIG = {
    "dataset": "tongpython/cat-and-dog",

    "workspace": None,

    "image_extensions": ["*.jpg", "*.jpeg", "*.png"],

    "proxy": {
        "backbone": "efficientnet_b0",
        "preprocessing": "stretch",
        "image_size": 224,
        "batch_size": 128,
        "num_workers": 0,
        "val_size": 0.2,
        "classifier_C": 0.1,
    },

    "dedup": {
        "default_hamming_threshold": 4,
        "lsh_bands": 8,
    },

    "score": {
        "retention_penalty": 0.10,
        "imbalance_penalty": 0.05,
    },

    "selection": {
        "min_delta_f1": 0.002,
        "min_retention_pct": 95.0,
        "max_imbalance_shift": 0.01,
        "default_policy": "minimal_safety",
        "stability_seeds": [21, 42, 84],
    },
}

DEVICE = get_device()

WORKSPACE = resolve_workspace(
    workspace=CONFIG["workspace"],
    project_name="sweet_spot_threshold",
)

RESULT_DIR = WORKSPACE / "results"
FIG_DIR = RESULT_DIR / "figures"
FEATURE_DIR = WORKSPACE / "features"

ensure_dirs(WORKSPACE, RESULT_DIR, FIG_DIR, FEATURE_DIR)

def savefig(name):
    path = FIG_DIR / name
    savefig_module(path)
    print("Saved:", path)

print("Device:", DEVICE)
print("Workspace:", WORKSPACE)
print("Result directory:", RESULT_DIR)
print("Figure directory:", FIG_DIR)
print("Feature directory:", FEATURE_DIR)

Device: cuda
Workspace: /content/sweet_spot_threshold
Result directory: /content/sweet_spot_threshold/results
Figure directory: /content/sweet_spot_threshold/results/figures
Feature directory: /content/sweet_spot_threshold/features


# 2. Dataset Loading

The dataset is loaded from a public Kaggle source. The raw dataframe contains only the minimum required information for downstream processing:

- `path`: image file path;
- `label`: numeric class label;
- `label_name`: human-readable class name.

No cleaning is applied at this stage.

In [3]:
# =============================
# 2. DATASET LOADING
# =============================

DATA_ROOT = resolve_dataset_root(
    dataset_id=CONFIG["dataset"],
    local_root=None,
)

raw_df = build_raw_dataframe(
    root=DATA_ROOT,
    extensions=CONFIG["image_extensions"],
)

print("Dataset root:", DATA_ROOT)
print("Raw dataframe shape:", raw_df.shape)

display(raw_df.head())
display(summarize_class_distribution(raw_df))

display(Markdown(
    f"""
### Dataset loading summary

The dataset was loaded from the public Kaggle dataset `{CONFIG["dataset"]}`.

A total of **{len(raw_df):,}** labelled image files were found.

The raw dataframe is used as the base input for image-quality auditing. At this point, no image has been removed.
"""
))

Using Colab cache for faster access to the 'cat-and-dog' dataset.
Dataset root: /kaggle/input/cat-and-dog
Raw dataframe shape: (10028, 3)


,path,label,label_name
0,/kaggle/input/cat-and-dog/test_set/test_set/ca...,0,Cat
1,/kaggle/input/cat-and-dog/test_set/test_set/ca...,0,Cat
2,/kaggle/input/cat-and-dog/test_set/test_set/ca...,0,Cat
3,/kaggle/input/cat-and-dog/test_set/test_set/ca...,0,Cat
4,/kaggle/input/cat-and-dog/test_set/test_set/ca...,0,Cat


,class,count,percentage
0,Cat,5011,49.970084
1,Dog,5017,50.029916



### Dataset loading summary

The dataset was loaded from the public Kaggle dataset `tongpython/cat-and-dog`.

A total of **10,028** labelled image files were found.

The raw dataframe is used as the base input for image-quality auditing. At this point, no image has been removed.


# 3. Image-Quality Audit

This section computes image-quality metrics for every image.

The audit stage measures possible data-quality problems, but it does not decide whether images should be removed. This separation is important because measurement and cleaning decisions are different methodological steps.

The audit includes:

- blur using Laplacian variance;
- grayscale tolerance;
- near-monochrome dark/bright ratios;
- image resolution and aspect ratio;
- entropy;
- saturation and chromaticity;
- center saliency;
- compression artifact score;
- perceptual hash for near-duplicate detection.

In [ ]:
# =============================
# 3. IMAGE AUDIT
# =============================

AUDIT_PATH = RESULT_DIR / "audit_metrics.csv"

if AUDIT_PATH.exists():
    audit_df = pd.read_csv(AUDIT_PATH)
    print("Loaded audit cache:", AUDIT_PATH)
else:
    audit_df = audit_dataframe(
        raw_df,
        compute_hash=True,
    )
    audit_df.to_csv(AUDIT_PATH, index=False)
    print("Saved audit cache:", AUDIT_PATH)

print("Audit dataframe shape:", audit_df.shape)
display(audit_df.head())

metric_summary = describe_audit_metrics(audit_df)
display(metric_summary.round(4))

display(Markdown(
    """
### Audit-stage interpretation

The audit dataframe contains image-quality indicators for the full dataset.
No cleaning rule has been applied yet.

These metrics will later be used to evaluate whether a cleaning threshold is technically justified and whether it improves downstream proxy validation performance.
"""
))

# 4. Near-Duplicate Audit

Perceptual hashing is used to identify near-duplicate images.

This is treated as a potential data leakage issue rather than a pure performance-optimization rule. If near-duplicate images appear across train and test sets, test performance may become overly optimistic.

In [ ]:
# =============================
# 4. NEAR-DUPLICATE AUDIT
# =============================

DEDUP_DEFAULT = CONFIG["dedup"]["default_hamming_threshold"]
DEDUP_SUMMARY_PATH = RESULT_DIR / "dedup_summary.csv"

dedup_records = []

for threshold in [0, 2, 4, 6, 8]:
    dedup_candidate_df = mark_near_duplicates(
        audit_df,
        hamming_threshold=threshold,
        hash_col="phash",
        bands=CONFIG["dedup"]["lsh_bands"],
    )

    duplicate_count = int(dedup_candidate_df["is_duplicate"].sum())
    duplicate_pct = duplicate_count / len(dedup_candidate_df) * 100

    dedup_records.append(
        {
            "hamming_threshold": threshold,
            "duplicates": duplicate_count,
            "duplicate_pct": duplicate_pct,
            "kept": int((~dedup_candidate_df["is_duplicate"]).sum()),
            "kept_pct": 100 - duplicate_pct,
            "n_clusters": int(dedup_candidate_df["duplicate_cluster"].nunique()),
        }
    )

    if threshold == DEDUP_DEFAULT:
        audit_df = dedup_candidate_df.copy()

dedup_df = pd.DataFrame(dedup_records)
dedup_df.to_csv(DEDUP_SUMMARY_PATH, index=False)
audit_df.to_csv(AUDIT_PATH, index=False)

display(dedup_df.round(4))

duplicate_count = int(audit_df["is_duplicate"].sum())
duplicate_pct = duplicate_count / len(audit_df) * 100

display(Markdown(
    f"""
### Duplicate-audit interpretation

Using perceptual hashing with the default Hamming threshold of **{DEDUP_DEFAULT}**, the audit detected **{duplicate_count:,}** near-duplicate images, representing approximately **{duplicate_pct:.2f}%** of the dataset.

Near-duplicate removal is considered a conservative evaluation safeguard. Even when the number of duplicates is small, removing them before train/test splitting can reduce the risk of leakage.
"""
))

# 5. Metric Distribution Analysis

This section visualizes the distribution of key audit metrics.

The goal is to understand whether potential cleaning thresholds affect only extreme low-quality tails or whether they cut deeply into normal image regions.

In [ ]:
# =============================
# 5. METRIC DISTRIBUTION ANALYSIS
# =============================

plot_specs = [
    ("blur_laplacian", "Blur Score: Laplacian Variance", [20, 30, 50, 75, 100]),
    ("min_side", "Minimum Image Resolution", [32, 64, 96, 128]),
    ("near_mono_ratio", "Near-Monochrome Ratio", [0.90, 0.95, 0.98]),
    ("entropy", "Image Entropy", [2.0, 3.0, 4.0]),
    ("mean_sat", "Mean Saturation", [5, 10, 20]),
    ("brightness_std", "Brightness Standard Deviation", [10, 15, 20]),
    ("aspect_extremity", "Aspect Ratio Extremity", [3.0, 4.0, 5.0]),
    ("center_saliency_ratio", "Center Saliency Ratio", [0.15, 0.20, 0.25]),
    ("compression_artifact", "Compression Artifact Score", [1.5, 2.0, 2.5, 3.0]),
]

for metric, title, thresholds in plot_specs:
    plot_metric_distribution(
        audit_df,
        metric=metric,
        thresholds=thresholds,
        title=title,
        save_path=FIG_DIR / f"metric_distribution_{metric}.png",
    )

display(Markdown(
    """
### Metric-distribution interpretation

The metric distributions help determine whether candidate thresholds are conservative or aggressive.
A reasonable cleaning threshold should mainly affect extreme low-quality tails, rather than remove a large portion of visually valid images.
"""
))

In [ ]:
# =============================
# 5.1 METRIC CORRELATION
# =============================

correlation_metrics = [
    "blur_laplacian",
    "min_side",
    "aspect_extremity",
    "gray_diff_mean",
    "near_mono_ratio",
    "dark_ratio",
    "bright_ratio",
    "entropy",
    "mean_sat",
    "chroma_mean",
    "center_saliency_ratio",
    "compression_artifact",
    "brightness_std",
]

plot_metric_correlation_heatmap(
    audit_df,
    metrics=correlation_metrics,
    save_path=FIG_DIR / "metric_correlation_heatmap.png",
)

display(Markdown(
    """
### Metric-correlation interpretation

The correlation heatmap helps identify overlapping quality signals.
For example, near-monochrome, entropy, brightness variance, and saturation may partially describe similar low-information images.
This supports the use of combined soft flags rather than treating every metric as an independent hard filter.
"""
))

# 6. Extreme-Sample Visual Audit

Numerical metrics alone are not sufficient for data cleaning.  
This section inspects visually extreme samples for selected metrics.

The purpose is to check whether low metric values correspond to genuinely invalid images or merely difficult but still valid samples.

In [ ]:
# =============================
# 6. EXTREME-SAMPLE VISUAL AUDIT
# =============================

def plot_extreme_samples(
    df,
    metric,
    ascending=True,
    n=12,
    title=None,
    save_name=None,
):
    sub = (
        df
        .dropna(subset=[metric])
        .sort_values(metric, ascending=ascending)
        .head(n)
    )

    if len(sub) == 0:
        print("No samples to display for:", metric)
        return

    cols = min(6, len(sub))
    rows = int(np.ceil(len(sub) / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 3.0))

    if rows == 1 and cols == 1:
        axes = np.array([axes])

    axes = np.asarray(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sub.iterrows()):
        try:
            img = Image.open(row["path"]).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "Unreadable", ha="center", va="center")

        ax.axis("off")
        ax.set_title(
            f'{row["label_name"]}\n{metric}={row[metric]:.3g}',
            fontsize=8,
        )

    for ax in axes[len(sub):]:
        ax.axis("off")

    plt.suptitle(title or f"Extreme samples: {metric}", fontsize=14, fontweight="bold")

    if save_name is not None:
        savefig(save_name)

    plt.show()


extreme_specs = [
    ("blur_laplacian", True, "Lowest Blur Scores", "extreme_low_blur.png"),
    ("min_side", True, "Smallest Images", "extreme_smallest_images.png"),
    ("entropy", True, "Lowest Entropy Images", "extreme_low_entropy.png"),
    ("near_mono_ratio", False, "Highest Near-Monochrome Ratio", "extreme_high_near_mono.png"),
    ("compression_artifact", False, "Highest Compression Artifact Scores", "extreme_high_compression.png"),
    ("center_saliency_ratio", True, "Lowest Center Saliency Ratio", "extreme_low_center_saliency.png"),
]

for metric, ascending, title, save_name in extreme_specs:
    plot_extreme_samples(
        audit_df,
        metric=metric,
        ascending=ascending,
        n=12,
        title=title,
        save_name=save_name,
    )

display(Markdown(
    """
### Visual-audit interpretation

The visual audit checks whether extreme metric values actually correspond to unusable images.
This step is important because some images may look unusual, blurry, low-contrast, or off-center while still containing valid class information.

Therefore, these metrics should not automatically become hard filters unless proxy validation and visual inspection both support the removal.
"""
))

# 7. Proxy Feature Extraction

To evaluate cleaning thresholds efficiently, this notebook uses a fixed proxy model:

- EfficientNet-B0 as a pretrained feature extractor;
- Logistic Regression as a lightweight classifier.

The proxy model is not the final project model. It is only used to test whether cleaning thresholds improve downstream feature separability.

In [ ]:
# =============================
# 7. PROXY FEATURE EXTRACTION
# =============================

FEATURE_PATH = FEATURE_DIR / "X_proxy_efficientnet_b0.npy"
LABEL_PATH = FEATURE_DIR / "y_proxy.npy"

proxy_transform = get_hybrid_transform(
    mode=CONFIG["proxy"]["preprocessing"],
    image_size=CONFIG["proxy"]["image_size"],
    train=False,
)

if FEATURE_PATH.exists() and LABEL_PATH.exists():
    X_all = np.load(FEATURE_PATH)
    y_all = np.load(LABEL_PATH)

    if len(X_all) != len(audit_df):
        print("Cached feature size does not match audit dataframe. Recomputing features.")
        X_all, y_all = extract_features(
            df=audit_df,
            transform=proxy_transform,
            backbone_name=CONFIG["proxy"]["backbone"],
            batch_size=CONFIG["proxy"]["batch_size"],
            device=DEVICE,
            num_workers=CONFIG["proxy"]["num_workers"],
        )
        np.save(FEATURE_PATH, X_all)
        np.save(LABEL_PATH, y_all)
else:
    X_all, y_all = extract_features(
        df=audit_df,
        transform=proxy_transform,
        backbone_name=CONFIG["proxy"]["backbone"],
        batch_size=CONFIG["proxy"]["batch_size"],
        device=DEVICE,
        num_workers=CONFIG["proxy"]["num_workers"],
    )
    np.save(FEATURE_PATH, X_all)
    np.save(LABEL_PATH, y_all)

print("Feature matrix:", X_all.shape)
print("Labels:", y_all.shape)

display(Markdown(
    f"""
### Proxy feature extraction summary

EfficientNet-B0 extracted a feature matrix with shape **{X_all.shape}**.

These features are cached as `.npy` files so that threshold experiments can be repeated without repeatedly running the CNN backbone.
"""
))

# 8. Fixed Proxy Validation Protocol

A fair threshold experiment must avoid making the validation set easier.

Therefore:

- The validation set is fixed.
- Cleaning masks are applied only to the proxy training set.
- The validation set remains unchanged across all candidates.

This design tests whether training on cleaned data improves validation performance, rather than improving metrics by removing difficult validation samples.

In [ ]:
# =============================
# 8. FIXED PROXY VALIDATION PROTOCOL
# =============================

VALID_BASE_MASK = ~audit_df["is_corrupted"].fillna(False).astype(bool)

valid_indices = audit_df[VALID_BASE_MASK].index.to_numpy()
valid_labels = audit_df.loc[valid_indices, "label"].values

train_idx, val_idx = train_test_split(
    valid_indices,
    test_size=CONFIG["proxy"]["val_size"],
    stratify=valid_labels,
    random_state=SEED,
)

print("Valid candidate images:", len(valid_indices))
print("Proxy train size:", len(train_idx))
print("Proxy validation size:", len(val_idx))

display(audit_df.loc[train_idx, "label_name"].value_counts().to_frame("train_count"))
display(audit_df.loc[val_idx, "label_name"].value_counts().to_frame("validation_count"))


def make_proxy_split(seed):
    train_indices, val_indices = train_test_split(
        valid_indices,
        test_size=CONFIG["proxy"]["val_size"],
        stratify=valid_labels,
        random_state=seed,
    )
    return train_indices, val_indices


def evaluate_cleaning_mask(
    mask,
    name,
    threshold,
    train_indices,
    val_indices,
    seed,
):
    mask = pd.Series(mask, index=audit_df.index).fillna(False).astype(bool)
    mask = mask & VALID_BASE_MASK

    train_keep_idx = np.array([idx for idx in train_indices if mask.loc[idx]])
    val_keep_idx = val_indices

    valid_retention_frac = mask.loc[valid_indices].mean()
    train_retention_frac = len(train_keep_idx) / len(train_indices)

    if len(train_keep_idx) < 100 or len(np.unique(y_all[train_keep_idx])) < 2:
        return {
            "filter": name,
            "threshold": threshold,
            "seed": seed,
            "accuracy": np.nan,
            "f1_macro": np.nan,
            "precision_macro": np.nan,
            "recall_macro": np.nan,
            "n_train": len(train_keep_idx),
            "n_val": len(val_keep_idx),
            "valid_retention_pct": 100 * valid_retention_frac,
            "train_retention_pct": 100 * train_retention_frac,
            "positive_ratio_after": np.nan,
            "imbalance_shift": np.nan,
            "score": np.nan,
        }

    clf = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    C=CONFIG["proxy"]["classifier_C"],
                    max_iter=1000,
                    random_state=seed,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    clf.fit(X_all[train_keep_idx], y_all[train_keep_idx])
    preds = clf.predict(X_all[val_keep_idx])

    acc = accuracy_score(y_all[val_keep_idx], preds)
    f1 = f1_score(y_all[val_keep_idx], preds, average="macro")
    prec = precision_score(y_all[val_keep_idx], preds, average="macro", zero_division=0)
    rec = recall_score(y_all[val_keep_idx], preds, average="macro", zero_division=0)

    positive_ratio_after = audit_df.loc[train_keep_idx, "label"].mean()
    base_ratio = audit_df.loc[train_indices, "label"].mean()
    imbalance_shift = abs(positive_ratio_after - base_ratio)

    score = (
        f1
        - CONFIG["score"]["retention_penalty"] * (1 - train_retention_frac)
        - CONFIG["score"]["imbalance_penalty"] * imbalance_shift
    )

    return {
        "filter": name,
        "threshold": threshold,
        "seed": seed,
        "accuracy": acc,
        "f1_macro": f1,
        "precision_macro": prec,
        "recall_macro": rec,
        "n_train": len(train_keep_idx),
        "n_val": len(val_keep_idx),
        "valid_retention_pct": 100 * valid_retention_frac,
        "train_retention_pct": 100 * train_retention_frac,
        "positive_ratio_after": positive_ratio_after,
        "imbalance_shift": imbalance_shift,
        "score": score,
    }


baseline_mask = VALID_BASE_MASK.copy()

baseline_result = evaluate_cleaning_mask(
    baseline_mask,
    name="baseline_valid_only",
    threshold="none",
    train_indices=train_idx,
    val_indices=val_idx,
    seed=SEED,
)

display(pd.DataFrame([baseline_result]).round(4))

display(Markdown(
    """
### Proxy-evaluation protocol summary

The baseline keeps all valid, readable images.
Each candidate cleaning rule is applied only to the proxy training subset, while the validation subset remains fixed.

This avoids biased validation results caused by filtering out difficult validation images.
"""
))

# 9. Single-Metric Threshold Sweep

This section evaluates each image-quality metric independently.

The purpose is exploratory: it identifies whether any single metric has a strong relationship with downstream proxy performance.

A strong threshold should ideally:

- improve validation macro F1;
- retain most of the training data;
- avoid shifting class balance;
- remove samples that are visually justifiable.

In [ ]:
# =============================
# 9. SINGLE-METRIC THRESHOLD SWEEP
# =============================

def m_blur(t):
    return VALID_BASE_MASK & (audit_df["blur_laplacian"] >= t)

def m_gray(t):
    return VALID_BASE_MASK & (audit_df["gray_diff_mean"] >= t)

def m_dark(t):
    return VALID_BASE_MASK & (audit_df["dark_ratio"] <= t)

def m_bright(t):
    return VALID_BASE_MASK & (audit_df["bright_ratio"] <= t)

def m_near_mono(t):
    return VALID_BASE_MASK & (audit_df["near_mono_ratio"] <= t)

def m_aspect(t):
    return VALID_BASE_MASK & (audit_df["aspect_extremity"] <= t)

def m_min_res(t):
    return VALID_BASE_MASK & (audit_df["min_side"] >= t)

def m_entropy_min(t):
    return VALID_BASE_MASK & (audit_df["entropy"] >= t)

def m_entropy_max(t):
    return VALID_BASE_MASK & (audit_df["entropy"] <= t)

def m_sat_min(t):
    return VALID_BASE_MASK & (audit_df["mean_sat"] >= t)

def m_sat_max(t):
    return VALID_BASE_MASK & (audit_df["mean_sat"] <= t)

def m_chroma(t):
    return VALID_BASE_MASK & (audit_df["chroma_mean"] >= t)

def m_saliency(t):
    return VALID_BASE_MASK & (audit_df["center_saliency_ratio"] >= t)

def m_compression(t):
    return VALID_BASE_MASK & (audit_df["compression_artifact"] <= t)

def m_brightness_min(t):
    return VALID_BASE_MASK & (audit_df["brightness_std"] >= t)

def m_brightness_max(t):
    return VALID_BASE_MASK & (audit_df["brightness_std"] <= t)


FILTER_SPECS = [
    ("blur_laplacian_min", [20, 30, 50, 75, 100], m_blur),
    ("gray_diff_mean_min", [0, 1, 2, 5, 10, 15], m_gray),
    ("dark_ratio_max", [0.90, 0.95, 0.98, 0.99], m_dark),
    ("bright_ratio_max", [0.90, 0.95, 0.98, 0.99], m_bright),
    ("near_mono_ratio_max", [0.90, 0.95, 0.98, 0.99], m_near_mono),
    ("aspect_extremity_max", [2.5, 3.0, 4.0, 5.0, 8.0], m_aspect),
    ("min_side_min", [32, 48, 64, 96, 128], m_min_res),
    ("entropy_min", [2.0, 3.0, 4.0, 5.0, 5.5, 6.0], m_entropy_min),
    ("entropy_max", [7.0, 7.3, 7.5, 7.8, 8.0], m_entropy_max),
    ("mean_sat_min", [0, 5, 10, 20, 30, 50], m_sat_min),
    ("mean_sat_max", [180, 200, 220, 240, 255], m_sat_max),
    ("chroma_mean_min", [0, 2, 5, 10, 15, 20], m_chroma),
    ("center_saliency_ratio_min", [0.10, 0.15, 0.20, 0.25, 0.30], m_saliency),
    ("compression_artifact_max", [1.2, 1.5, 2.0, 2.5, 3.0, 5.0], m_compression),
    ("brightness_std_min", [5, 10, 15, 20, 30, 40], m_brightness_min),
    ("brightness_std_max", [70, 85, 100, 120, 160, 255], m_brightness_max),
]

SWEEP_PATH = RESULT_DIR / "single_metric_sweep_fixed_validation.csv"

records = [baseline_result]

for name, thresholds, mask_fn in FILTER_SPECS:
    for threshold in thresholds:
        mask = mask_fn(threshold)
        result = evaluate_cleaning_mask(
            mask,
            name=name,
            threshold=threshold,
            train_indices=train_idx,
            val_indices=val_idx,
            seed=SEED,
        )
        records.append(result)

sweep_df = pd.DataFrame(records)
sweep_df["delta_f1_vs_baseline"] = sweep_df["f1_macro"] - baseline_result["f1_macro"]
sweep_df["train_removed_pct"] = 100 - sweep_df["train_retention_pct"]

sweep_df.to_csv(SWEEP_PATH, index=False)

display(
    sweep_df
    .sort_values("score", ascending=False)
    .head(20)
    .round(4)
)

display(Markdown(
    """
### Single-metric sweep interpretation

Each threshold is evaluated independently.
A high F1-score alone is not sufficient: a useful threshold must also retain data and avoid class-balance distortion.

Very small F1 changes should be interpreted cautiously because they may correspond to only a few validation images.
"""
))

In [ ]:
# =============================
# 9.1 SINGLE-METRIC SWEEP VISUALIZATION
# =============================

best_single_df = (
    sweep_df[sweep_df["filter"] != "baseline_valid_only"]
    .dropna(subset=["score"])
    .sort_values("score", ascending=False)
    .groupby("filter", as_index=False)
    .head(1)
    .sort_values("score", ascending=False)
)

display(best_single_df.round(4))

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=sweep_df[sweep_df["filter"] != "baseline_valid_only"],
    x="train_retention_pct",
    y="f1_macro",
    hue="filter",
    s=90,
)
plt.axhline(
    baseline_result["f1_macro"],
    linestyle="--",
    color="black",
    label="Baseline F1",
)
plt.title("Single-Metric Threshold Sweep: Macro F1 vs Training Retention", fontweight="bold")
plt.xlabel("Training Retention (%)")
plt.ylabel("Validation Macro F1")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(axis="both", linestyle="--", alpha=0.3)
savefig("single_metric_f1_vs_train_retention.png")
plt.show()

plt.figure(figsize=(12, 7))
sns.barplot(
    data=best_single_df,
    x="score",
    y="filter",
)
plt.title("Best Threshold per Metric by Penalized Score", fontweight="bold")
plt.xlabel("Penalized score")
plt.ylabel("Metric")
savefig("best_single_thresholds.png")
plt.show()

top_single = best_single_df.iloc[0]

display(Markdown(
    f"""
### Single-metric sweep finding

The strongest single-metric candidate is **{top_single["filter"]} = {top_single["threshold"]}**, with validation macro F1 of **{top_single["f1_macro"]:.4f}**.

Its F1 change relative to the valid-only baseline is **{top_single["delta_f1_vs_baseline"]:+.4f}**, while retaining **{top_single["train_retention_pct"]:.2f}%** of the proxy training set.

This result is exploratory. A single metric should not be selected as the final cleaning policy unless it provides a practically meaningful improvement.
"""
))

# 10. Cleaning Preset Evaluation

Single-metric thresholds are useful for exploration, but final cleaning usually combines multiple rules.

This section evaluates complete cleaning presets:

- `valid_only_no_dedup`: remove only corrupted/unreadable images;
- `minimal_safety`: remove corrupted images and near-duplicates;
- `conservative`: light quality filtering;
- `balanced`: moderate quality filtering;
- `strict`: aggressive quality filtering.

The final policy will not be selected only by raw F1-score. It must also satisfy retention and class-balance requirements.

In [ ]:
# =============================
# 10. CLEANING PRESET EVALUATION
# =============================

CLEANING_PRESETS = {
    "valid_only_no_dedup": {
        "remove_corrupted": True,
        "remove_duplicates": False,
        "max_soft_flags": 99,
    },

    "minimal_safety": {
        "remove_corrupted": True,
        "remove_duplicates": True,
        "max_soft_flags": 99,
    },

    "conservative": {
        "remove_corrupted": True,
        "remove_duplicates": True,
        "blur_laplacian_min": 20,
        "min_side_min": 32,
        "aspect_extremity_max": 8.0,
        "dark_ratio_max": 0.99,
        "bright_ratio_max": 0.99,
        "entropy_near_mono_max": 2.0,
        "gray_diff_mean_min": 0,
        "entropy_min": 2.0,
        "entropy_max": 8.0,
        "mean_sat_min": 0,
        "mean_sat_max": 255,
        "chroma_mean_min": 0,
        "center_saliency_ratio_min": 0.10,
        "compression_artifact_max": 5.0,
        "brightness_std_min": 5,
        "brightness_std_max": 255,
        "max_soft_flags": 3,
    },

    "balanced": {
        "remove_corrupted": True,
        "remove_duplicates": True,
        "blur_laplacian_min": 50,
        "min_side_min": 64,
        "aspect_extremity_max": 5.0,
        "dark_ratio_max": 0.95,
        "bright_ratio_max": 0.95,
        "entropy_near_mono_max": 3.0,
        "gray_diff_mean_min": 1,
        "entropy_min": 3.0,
        "entropy_max": 7.8,
        "mean_sat_min": 5,
        "mean_sat_max": 240,
        "chroma_mean_min": 2,
        "center_saliency_ratio_min": 0.15,
        "compression_artifact_max": 3.0,
        "brightness_std_min": 10,
        "brightness_std_max": 160,
        "max_soft_flags": 2,
    },

    "strict": {
        "remove_corrupted": True,
        "remove_duplicates": True,
        "blur_laplacian_min": 75,
        "min_side_min": 96,
        "aspect_extremity_max": 4.0,
        "dark_ratio_max": 0.95,
        "bright_ratio_max": 0.95,
        "entropy_near_mono_max": 4.0,
        "gray_diff_mean_min": 2,
        "entropy_min": 4.0,
        "entropy_max": 7.5,
        "mean_sat_min": 10,
        "mean_sat_max": 230,
        "chroma_mean_min": 5,
        "center_saliency_ratio_min": 0.20,
        "compression_artifact_max": 2.5,
        "brightness_std_min": 15,
        "brightness_std_max": 120,
        "max_soft_flags": 1,
    },
}

PRESET_PATH = RESULT_DIR / "cleaning_preset_results_fixed_validation.csv"

preset_records = []

for preset_name, preset_cfg in CLEANING_PRESETS.items():
    preset_mask = build_cleaning_mask(audit_df, preset_cfg)
    result = evaluate_cleaning_mask(
        preset_mask,
        name=preset_name,
        threshold="preset",
        train_indices=train_idx,
        val_indices=val_idx,
        seed=SEED,
    )
    preset_records.append(result)

preset_df = pd.DataFrame(preset_records)
preset_df["delta_f1_vs_baseline"] = preset_df["f1_macro"] - baseline_result["f1_macro"]
preset_df["train_removed_pct"] = 100 - preset_df["train_retention_pct"]

preset_df.to_csv(PRESET_PATH, index=False)

display(
    preset_df
    .sort_values("score", ascending=False)
    .round(4)
)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=preset_df.sort_values("f1_macro", ascending=False),
    x="filter",
    y="f1_macro",
)
plt.axhline(
    baseline_result["f1_macro"],
    linestyle="--",
    color="black",
    label="Baseline F1",
)
plt.title("Cleaning Preset Comparison by Validation Macro F1", fontweight="bold")
plt.xlabel("Cleaning preset")
plt.ylabel("Validation macro F1")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.35)
plt.legend()
savefig("cleaning_preset_comparison.png")
plt.show()

display(Markdown(
    """
### Preset-level interpretation

Preset evaluation is more realistic than single-metric evaluation because cleaning policies usually combine multiple quality checks.

A preset is considered useful only if it improves proxy validation performance by a meaningful margin while retaining most training samples and preserving class balance.
"""
))

# 11. Stability Check Across Multiple Splits

A single train/validation split can produce unstable conclusions, especially when F1 differences are small.

This section repeats the preset comparison across multiple random seeds. The final decision uses mean performance rather than relying only on one split.

In [ ]:
# =============================
# 11. STABILITY CHECK ACROSS MULTIPLE SPLITS
# =============================

STABILITY_PATH = RESULT_DIR / "cleaning_preset_stability.csv"

stability_records = []

for seed in CONFIG["selection"]["stability_seeds"]:
    split_train_idx, split_val_idx = make_proxy_split(seed)

    split_baseline = evaluate_cleaning_mask(
        baseline_mask,
        name="baseline_valid_only",
        threshold="none",
        train_indices=split_train_idx,
        val_indices=split_val_idx,
        seed=seed,
    )

    for preset_name, preset_cfg in CLEANING_PRESETS.items():
        preset_mask = build_cleaning_mask(audit_df, preset_cfg)

        result = evaluate_cleaning_mask(
            preset_mask,
            name=preset_name,
            threshold="preset",
            train_indices=split_train_idx,
            val_indices=split_val_idx,
            seed=seed,
        )

        result["baseline_f1_for_seed"] = split_baseline["f1_macro"]
        result["delta_f1_vs_seed_baseline"] = result["f1_macro"] - split_baseline["f1_macro"]

        stability_records.append(result)

stability_df = pd.DataFrame(stability_records)
stability_df.to_csv(STABILITY_PATH, index=False)

stability_summary = (
    stability_df
    .groupby("filter")
    .agg(
        mean_f1=("f1_macro", "mean"),
        std_f1=("f1_macro", "std"),
        mean_delta_f1=("delta_f1_vs_seed_baseline", "mean"),
        std_delta_f1=("delta_f1_vs_seed_baseline", "std"),
        mean_train_retention_pct=("train_retention_pct", "mean"),
        mean_imbalance_shift=("imbalance_shift", "mean"),
        mean_score=("score", "mean"),
    )
    .reset_index()
    .sort_values("mean_score", ascending=False)
)

display(stability_summary.round(4))

display(Markdown(
    """
### Stability-check interpretation

The stability check reduces the risk of selecting a cleaning policy due to random split variation.
If a preset improves F1 only on one split but not consistently across seeds, it should not be treated as strong evidence.

The final policy selection is based on mean F1 improvement, retention, and class-balance stability.
"""
))

# 12. Final Cleaning Policy Selection

The final cleaning policy is selected using predefined practical criteria.

A threshold-based preset is eligible only if it satisfies all conditions:

1. Mean macro F1 improvement is at least `min_delta_f1`.
2. Mean training retention is at least `min_retention_pct`.
3. Mean class-balance shift is no greater than `max_imbalance_shift`.

If no quality-threshold preset satisfies these conditions, the notebook selects `minimal_safety`, which removes corrupted images and near-duplicates but avoids aggressive quality filtering.

In [ ]:
# =============================
# 12. FINAL CLEANING POLICY SELECTION
# =============================

MIN_DELTA_F1 = CONFIG["selection"]["min_delta_f1"]
MIN_RETENTION = CONFIG["selection"]["min_retention_pct"]
MAX_IMBALANCE_SHIFT = CONFIG["selection"]["max_imbalance_shift"]
DEFAULT_POLICY = CONFIG["selection"]["default_policy"]

candidate_df = stability_summary.copy()

candidate_df["meets_practical_gain"] = candidate_df["mean_delta_f1"] >= MIN_DELTA_F1
candidate_df["meets_retention"] = candidate_df["mean_train_retention_pct"] >= MIN_RETENTION
candidate_df["meets_balance"] = candidate_df["mean_imbalance_shift"] <= MAX_IMBALANCE_SHIFT

eligible_df = candidate_df[
    candidate_df["meets_practical_gain"]
    & candidate_df["meets_retention"]
    & candidate_df["meets_balance"]
].copy()

if len(eligible_df) > 0:
    selected_row = eligible_df.sort_values("mean_score", ascending=False).iloc[0]
    selection_rule = (
        "Selected the highest-scoring preset among candidates that met practical F1 gain, "
        "retention, and class-balance criteria."
    )
else:
    selected_row = candidate_df[candidate_df["filter"] == DEFAULT_POLICY].iloc[0]
    selection_rule = (
        "No quality-threshold preset achieved a practical and stable F1 improvement under "
        "the predefined criteria. The minimal-safety policy was selected to control corrupted "
        "and near-duplicate samples without aggressive quality filtering."
    )

best_preset_name = selected_row["filter"]
best_config = CLEANING_PRESETS[best_preset_name]

final_clean_df, removed_df = apply_cleaning(
    audit_df,
    best_config,
)

cleaning_summary = summarize_cleaning(
    raw_df=audit_df,
    clean_df=final_clean_df,
    removed_df=removed_df,
)

retention_summary = evaluate_cleaning_retention(
    raw_df=audit_df,
    clean_df=final_clean_df,
)

display(Markdown(f"### Selected cleaning policy: `{best_preset_name}`"))

display(pd.DataFrame([selected_row]).round(4))
display(cleaning_summary.round(2))
display(pd.DataFrame([retention_summary]).round(4))

plot_before_after_cleaning(
    cleaning_summary,
    save_path=FIG_DIR / "before_after_cleaning_class_counts.png",
)

display(Markdown(
    f"""
### Final-selection interpretation

{selection_rule}

The selected policy is **`{best_preset_name}`**.
Its mean F1 change relative to the baseline across stability seeds is **{selected_row["mean_delta_f1"]:+.4f}**.

This final decision is based on practical evidence rather than a mechanical ranking of raw F1-score.
A cleaning policy is only justified when its improvement is large enough to offset the cost of removing valid data.
"""
))

# 13. Removal-Reason Analysis

After selecting the final policy, the removed images are analyzed by removal reason.

This verifies whether the selected policy removes images for technically meaningful reasons.

In [ ]:
# =============================
# 13. REMOVAL-REASON ANALYSIS
# =============================

reason_counts = summarize_removal_reasons(removed_df)
display(reason_counts.round(2))

if len(reason_counts) > 0:
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=reason_counts.head(10),
        x="count",
        y="reason",
    )
    plt.title("Top Removal Reasons", fontweight="bold")
    plt.xlabel("Number of removed images")
    plt.ylabel("Reason")
    savefig("removed_reasons.png")
    plt.show()

    for reason in reason_counts["reason"].head(5):
        plot_removed_examples(
            removed_df,
            reason=reason,
            n=8,
            save_path=FIG_DIR / f"removed_examples_{str(reason).replace(' ', '_')}.png",
        )
else:
    display(Markdown(
        """
### Removal-reason analysis

No images were removed by the selected cleaning policy.
This is acceptable only if the selected policy is intentionally audit-only or minimal-cleaning.
"""
    ))

display(Markdown(
    """
### Removal-reason interpretation

The removal-reason table checks whether the selected cleaning policy removes images for justifiable technical reasons, such as corruption, near-duplication, blur, undersizing, or extreme low-information characteristics.

This step prevents the cleaning process from becoming a purely numerical optimization exercise.
"""
))

# 14. Save Final Outputs

This section saves the recommended cleaning configuration and the final clean image list.

These files can be used by the main training pipeline.

In [ ]:
# =============================
# 14. SAVE FINAL OUTPUTS
# =============================

final_clean_path = RESULT_DIR / "final_clean_images.csv"
final_config_path = RESULT_DIR / "recommended_cleaning_config.json"

final_clean_df[["path", "label", "label_name"]].to_csv(final_clean_path, index=False)

final_report = {
    "version": "sweet_spot_threshold_final_research_grade",
    "dataset": CONFIG["dataset"],
    "proxy_model": "EfficientNet-B0 fixed feature extractor + Logistic Regression",

    "selection_rule": selection_rule,

    "selection_criteria": {
        "min_delta_f1": MIN_DELTA_F1,
        "min_train_retention_pct": MIN_RETENTION,
        "max_imbalance_shift": MAX_IMBALANCE_SHIFT,
        "fixed_validation_set": True,
        "validation_filtering": "No candidate cleaning rule is applied to the proxy validation set.",
        "stability_seeds": CONFIG["selection"]["stability_seeds"],
    },

    "selected_preset": best_preset_name,

    "evaluated_rules": {
        "hard_filters_tested": [
            "corrupted image",
            "near duplicate image",
            "blur_laplacian_min",
            "min_side_min",
            "aspect_extremity_max",
            "near_mono_ratio_max",
            "dark_ratio_max with entropy condition",
            "bright_ratio_max with entropy condition",
        ],
        "soft_flags_tested": [
            "entropy_min",
            "mean_sat_min",
            "chroma_mean_min",
            "center_saliency_ratio_min",
            "compression_artifact_max",
            "brightness_std_min",
            "brightness_std_max",
        ],
    },

    "recommended_config": best_config,

    "results": {
        "selected_mean_f1": float(selected_row["mean_f1"]),
        "selected_std_f1": float(selected_row["std_f1"]) if not pd.isna(selected_row["std_f1"]) else None,
        "selected_mean_delta_f1": float(selected_row["mean_delta_f1"]),
        "selected_mean_train_retention_pct": float(selected_row["mean_train_retention_pct"]),
        "selected_mean_imbalance_shift": float(selected_row["mean_imbalance_shift"]),
        "selected_mean_score": float(selected_row["mean_score"]),

        "n_total": int(len(audit_df)),
        "n_clean": int(len(final_clean_df)),
        "n_removed": int(len(removed_df)),
        "cat_clean": int((final_clean_df["label"] == 0).sum()),
        "dog_clean": int((final_clean_df["label"] == 1).sum()),
    },
}

with open(final_config_path, "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=4)

print("Saved final clean image list:", final_clean_path)
print("Saved recommended cleaning config:", final_config_path)

display(final_clean_df.head())
display(pd.DataFrame([final_report["results"]]).T.rename(columns={0: "value"}))

# 15. Final Conclusion

This final section summarizes the result of the sweet-spot threshold experiment.

In [ ]:
# =============================
# 15. FINAL CONCLUSION
# =============================

summary_table = pd.DataFrame(
    [
        {
            "selected_preset": best_preset_name,
            "mean_f1": selected_row["mean_f1"],
            "std_f1": selected_row["std_f1"],
            "mean_delta_f1": selected_row["mean_delta_f1"],
            "mean_train_retention_pct": selected_row["mean_train_retention_pct"],
            "mean_imbalance_shift": selected_row["mean_imbalance_shift"],
            "n_total": len(audit_df),
            "n_clean": len(final_clean_df),
            "n_removed": len(removed_df),
        }
    ]
)

display(summary_table.round(4))

display(Markdown(
    f"""
## Final conclusion of the sweet-spot experiment

The sweet-spot threshold experiment evaluated whether image-quality based cleaning improves downstream classification under a fixed proxy validation protocol.

The selected cleaning policy is **`{best_preset_name}`**.

Across stability seeds, the selected policy achieves a mean macro F1 of **{selected_row["mean_f1"]:.4f}**, with a mean F1 change of **{selected_row["mean_delta_f1"]:+.4f}** relative to the valid-only baseline.

The predefined selection criteria require a practical F1 improvement of at least **{MIN_DELTA_F1:.4f}**, training retention of at least **{MIN_RETENTION:.1f}%**, and class-balance shift below **{MAX_IMBALANCE_SHIFT:.2f}**.

Therefore, this notebook does not assume that aggressive data cleaning is automatically beneficial.
Instead, it treats cleaning as an empirical decision based on proxy validation, data retention, class-balance stability, and visual audit evidence.

The final recommended cleaning configuration is saved to:

`{final_config_path}`

The final clean image list is saved to:

`{final_clean_path}`
"""
))

# 16. Files Generated

The notebook produces the following key outputs:

- `audit_metrics.csv`: image-quality audit metrics for the full dataset;
- `dedup_summary.csv`: perceptual-hash duplicate audit result;
- `single_metric_sweep_fixed_validation.csv`: single-metric threshold sweep result;
- `cleaning_preset_results_fixed_validation.csv`: preset-level evaluation on the main split;
- `cleaning_preset_stability.csv`: preset stability evaluation across multiple seeds;
- `recommended_cleaning_config.json`: final recommended cleaning policy;
- `final_clean_images.csv`: image list to be used by the main pipeline;
- `figures/`: all diagnostic plots.

In [ ]:
# =============================
# 16. FILE CHECK
# =============================

generated_files = [
    AUDIT_PATH,
    DEDUP_SUMMARY_PATH,
    SWEEP_PATH,
    PRESET_PATH,
    STABILITY_PATH,
    final_config_path,
    final_clean_path,
]

for path in generated_files:
    print(f"{Path(path).name}: {'OK' if Path(path).exists() else 'MISSING'}")

print("\nFigure directory:", FIG_DIR)
print("Number of figures:", len(list(FIG_DIR.glob('*.png'))))